## Ejercicios

1. Lee el archivo de deptmanagers.csv y haz un select de las 2 columnas que quieras
   
2. Añade una columna nueva al dataset de movies.json que sea el total generado por cada pelicula = US_Gross + Worldwide_Gross + DVD sales. Estas obteniendo algun nulo? Como lo puedes solucionar?
   
3. Selecciona todas las peliculas que tengan una nota mayor que 7 o igual en IMDB (IMDB_Rating), intenta hacerlo de todas las maneras que sepas.

In [ ]:
from pyspark.sql.functions import *

# ============================================================
# 1. Leer deptmanagers.csv y hacer un select de 2 columnas
# ============================================================

dept = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("deptmanagers.csv")
)

# Seleccionamos dos columnas cualquiera, por ejemplo dept_no y emp_no
dept_select = dept.select("dept_no", "emp_no")
dept_select.show()


# ============================================================
# 2. Nueva columna en movies.json = US_Gross + Worldwide_Gross + DVD_sales
# ============================================================

movies = (
    spark.read
        .option("multiline", True)
        .option("inferSchema", True)
        .json("movies.json")
)

movies.show(5)

# Creamos la columna total_revenue
movies_total = movies.withColumn(
    "total_revenue",
    col("US_Gross") + col("Worldwide_Gross") + col("DVD_sales")
)

movies_total.show(5)

# Pregunta: ¿Hay nulos?
# Sí, si alguna de estas columnas tiene null → el total también se vuelve null.
# Para evitarlo se usa coalesce o fillna.

movies_total_no_nulls = movies.withColumn(
    "total_revenue",
    coalesce(col("US_Gross"), lit(0)) +
    coalesce(col("Worldwide_Gross"), lit(0)) +
    coalesce(col("DVD_sales"), lit(0))
)

print("=== Sin nulos ===")
movies_total_no_nulls.show(5)


# ============================================================
# 3. Películas con IMDB_Rating >= 7 (varias formas)
# ============================================================

# --- Forma 1: filter con condición normal ---
pelis_7_a = movies.filter(col("IMDB_Rating") >= 7)

# --- Forma 2:where---
pelis_7_b = movies.where("IMDB_Rating >= 7")

# --- Forma 3: usando expr ---
pelis_7_c = movies.filter(expr("IMDB_Rating >= 7"))

# --- Forma 4: usando SQL (registrando vista) ---
movies.createOrReplaceTempView("movies_view")
pelis_7_d = spark.sql("""
    SELECT *
    FROM movies_view
    WHERE IMDB_Rating >= 7
""")

print("=== Pelis con nota >= 7 (filter) ===")
pelis_7_a.show()

print("=== Pelis con nota >= 7 (SQL) ===")
pelis_7_d.show()
